# Vehicle Speed Estimation — Google Colab

This notebook detects cars with **YOLOv8**, tracks them across frames, and estimates their
speed (km/h) as they cross two reference lines a known real-world distance apart.

**How to use**
1. Run the cells top to bottom, in order.
2. When you reach the "Upload video" cell, upload your file (e.g. `highway_mini.mp4`).
3. Adjust the settings in the **Configuration** cell if needed (line positions, real-world
   distance between the lines) to match your video.
4. Run the **Processing** cell — it will take a little while depending on video length.
5. Watch the output video and/or download it from the file panel on the left.

> Tip: Runtime → Change runtime type → **GPU** (T4) will make YOLO detection much faster.


## 1. Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless


## 2. Imports

In [ ]:
import os
import time
import math

import cv2
import pandas as pd
from ultralytics import YOLO
from IPython.display import HTML, display
from base64 import b64encode


## 3. Tracker (from `tracker.py` in the original project)\nA very simple centroid tracker: matches detections between frames by nearest center point.

In [ ]:
class Tracker:
    def __init__(self):
        # Store the center positions of the objects
        self.center_points = {}
        # Keep the count of the IDs, incremented for every new object detected
        self.id_count = 0

    def update(self, objects_rect):
        # Objects boxes and ids
        objects_bbs_ids = []

        # Get center point of new object
        for rect in objects_rect:
            x, y, w, h = rect
            cx = (x + x + w) // 2
            cy = (y + y + h) // 2

            # Find out if that object was detected already
            same_object_detected = False
            for id, pt in self.center_points.items():
                dist = math.hypot(cx - pt[0], cy - pt[1])
                if dist < 35:
                    self.center_points[id] = (cx, cy)
                    objects_bbs_ids.append([x, y, w, h, id])
                    same_object_detected = True
                    break

            # New object detected, assign it a new ID
            if not same_object_detected:
                self.center_points[self.id_count] = (cx, cy)
                objects_bbs_ids.append([x, y, w, h, self.id_count])
                self.id_count += 1

        # Clean the dictionary of center points to remove IDs no longer in use
        new_center_points = {}
        for obj_bb_id in objects_bbs_ids:
            _, _, _, _, object_id = obj_bb_id
            new_center_points[object_id] = self.center_points[object_id]
        self.center_points = new_center_points.copy()

        return objects_bbs_ids


## 4. Upload your video\nRun this cell, then click **Choose Files** and select your video (e.g. `highway_mini.mp4`).\n\nIf you'd rather not use the upload widget, you can instead mount Google Drive and point `VIDEO_PATH` in the next cell at a file already in your Drive.

In [ ]:
from google.colab import files

uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {VIDEO_PATH}")


## 5. Configuration

- `red_line_y` / `blue_line_y`: the y-pixel positions of the two reference lines, measured on
  the frame **after** it is resized to `FRAME_W x FRAME_H`.
- `real_distance_m`: the real-world distance (in meters) between the red and blue lines on the
  road. Getting this right matters a lot for accurate speed — measure it (or estimate it) from
  the actual road/camera setup.
- `offset`: how many pixels of tolerance to use when checking whether a vehicle's center has
  "touched" a line.

The defaults below match the original project's `highway.mp4` / `highway_mini.mp4` clips. If
your video has a different resolution, camera angle, or the cars appear at different heights in
frame, drag the lines around by changing these values (re-run the processing cell after) until
the red line sits where a car enters the measured stretch of road and the blue line sits where it
leaves it.


In [ ]:
FRAME_W, FRAME_H = 1020, 500

red_line_y = 198
blue_line_y = 268
offset = 6

real_distance_m = 10   # real-world distance between the two lines, in meters

# Only these YOLO/COCO classes will be treated as vehicles for tracking + speed
VEHICLE_CLASSES = {'car', 'truck', 'bus'}

OUTPUT_PATH = 'output.mp4'


## 6. Load the YOLOv8 model

In [ ]:
model = YOLO('yolov8s.pt')  # downloads weights automatically on first run
class_list = model.names    # dict: class id -> class name, straight from the model


## 7. Process the video\nDetects vehicles frame by frame, tracks them, measures the time each tracked vehicle takes to travel between the red and blue lines, and writes an annotated video to `output.mp4`.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

src_fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, src_fps, (FRAME_W, FRAME_H))

tracker = Tracker()

down = {}            # id -> timestamp when it crossed the red line, travelling down
up = {}              # id -> timestamp when it crossed the blue line, travelling up
counter_down = []    # ids already counted going down
counter_up = []      # ids already counted going up
speeds = {}          # id -> last computed speed (km/h), kept for display

count = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    count += 1
    frame = cv2.resize(frame, (FRAME_W, FRAME_H))

    results = model.predict(frame, verbose=False)
    boxes = results[0].boxes.data.detach().cpu().numpy()
    px = pd.DataFrame(boxes).astype("float")

    detections = []
    for _, row in px.iterrows():
        x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
        cls_id = int(row[5])
        cls_name = class_list[cls_id]
        if cls_name in VEHICLE_CLASSES:
            detections.append([x1, y1, x2, y2])

    bbox_id = tracker.update(detections)

    for bbox in bbox_id:
        x3, y3, x4, y4, obj_id = bbox
        cx = int(x3 + x4) // 2
        cy = int(y3 + y4) // 2

        # ---- travelling DOWN (red line first, then blue line) ----
        if red_line_y - offset < cy < red_line_y + offset:
            down[obj_id] = time.time()

        if obj_id in down and blue_line_y - offset < cy < blue_line_y + offset:
            if obj_id not in counter_down:
                counter_down.append(obj_id)
                elapsed = time.time() - down[obj_id]
                if elapsed > 0:
                    speed_kmh = (real_distance_m / elapsed) * 3.6
                    speeds[obj_id] = speed_kmh

        # ---- travelling UP (blue line first, then red line) ----
        if blue_line_y - offset < cy < blue_line_y + offset:
            up[obj_id] = time.time()

        if obj_id in up and red_line_y - offset < cy < red_line_y + offset:
            if obj_id not in counter_up:
                counter_up.append(obj_id)
                elapsed = time.time() - up[obj_id]
                if elapsed > 0:
                    speed_kmh = (real_distance_m / elapsed) * 3.6
                    speeds[obj_id] = speed_kmh

        # ---- draw annotations ----
        cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
        cv2.rectangle(frame, (x3, y3), (x4, y4), (0, 255, 0), 2)
        cv2.putText(frame, str(obj_id), (x3, y3), cv2.FONT_HERSHEY_COMPLEX, 0.6, (255, 255, 255), 1)
        if obj_id in speeds:
            label = f"{int(speeds[obj_id])} Km/h"
            cv2.putText(frame, label, (x4, y4), cv2.FONT_HERSHEY_COMPLEX, 0.8, (0, 255, 255), 2)

    # ---- overlay lines + counters ----
    cv2.rectangle(frame, (0, 0), (250, 90), (0, 255, 255), -1)
    cv2.line(frame, (0, red_line_y), (FRAME_W, red_line_y), (0, 0, 255), 2)
    cv2.putText(frame, 'Red Line', (10, red_line_y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.line(frame, (0, blue_line_y), (FRAME_W, blue_line_y), (255, 0, 0), 2)
    cv2.putText(frame, 'Blue Line', (10, blue_line_y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(frame, f'Going Down - {len(counter_down)}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(frame, f'Going Up - {len(counter_up)}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

    out.write(frame)

    if count % 30 == 0 or count == total_frames:
        pct = (count / total_frames * 100) if total_frames else 0
        print(f"Processed {count}/{total_frames} frames ({pct:.0f}%)")

cap.release()
out.release()

print(f"\nDone in {time.time() - start_time:.1f}s. Vehicles counted — down: {len(counter_down)}, up: {len(counter_up)}")
print(f"Annotated video saved to: {OUTPUT_PATH}")


## 8. Preview the result inline

In [ ]:
# Re-encode with ffmpeg for reliable inline playback in Colab
PLAYABLE_PATH = 'output_playable.mp4'
!ffmpeg -y -loglevel error -i {OUTPUT_PATH} -vcodec libx264 {PLAYABLE_PATH}

mp4 = open(PLAYABLE_PATH, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f"""
<video width=640 controls>
      <source src="{data_url}" type="video/mp4">
</video>
"""))


## 9. Download the output video

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)
